# الذكاء الاصطناعي في الهندسة المدنية — الفصل 11 Construction management
Companion notebook. Run all cells (Runtime → Run all). Every number in the book comes from this code.

In [ ]:
!pip -q install xgboost shap scikit-image statsmodels ucimlrepo
import os; os.makedirs('fig', exist_ok=True)

In [ ]:
%%writefile clauses.py
CLAUSES = {
"payment": [
"يدفع صاحب العمل للمقاول الدفعة المرحلية خلال ثلاثين يوماً من تاريخ اعتماد المطالبة",
"تُصرف الدفعة الختامية بعد صدور شهادة الاستلام النهائي وتسوية جميع الحسابات",
"يحتجز صاحب العمل نسبة خمسة بالمئة من كل دفعة ضماناً لحسن التنفيذ",
"يقدّم المقاول مطالبة شهرية مفصلة بالكميات المنفذة مرفقة بالمستندات المؤيدة",
"تُدفع الدفعة المقدمة مقابل كفالة بنكية غير مشروطة بنفس القيمة",
"في حال تأخر صاحب العمل عن الدفع يستحق المقاول فائدة عن مدة التأخير",
"تُستردّ الدفعة المقدمة على أقساط تُخصم من الدفعات المرحلية",
"لا تُعدّ أي دفعة مرحلية قبولاً للأعمال المنفذة",
"يعتمد المهندس قيمة الأعمال المنجزة خلال أربعة عشر يوماً من استلام الكشف",
"تُفرج المحتجزات بنسبة النصف عند الاستلام الأولي والباقي عند انتهاء فترة الصيانة",
"تُسدد قيمة المواد الموردة إلى الموقع بنسبة ثمانين بالمئة من قيمتها المعتمدة",
"يحق لصاحب العمل خصم أي مبالغ مستحقة له من الدفعات اللاحقة",
"تُصرف المستحقات بالدينار الأردني بموجب حوالة بنكية إلى حساب المقاول",
"يرفق المقاول مع كل كشف دفعة شهادة براءة ذمة من الضمان الاجتماعي",
"لا تُدفع قيمة الأعمال الإضافية إلا بعد اعتماد أمر التغيير خطياً",
"تُحسب قيمة الأعمال المنجزة وفق أسعار جدول الكميات المعتمد"],
"delay": [
"إذا تأخر المقاول عن إنجاز الأعمال تُفرض عليه غرامة يومية قدرها نصف بالألف من قيمة العقد",
"لا يتجاوز مجموع غرامات التأخير عشرة بالمئة من قيمة العقد",
"تُخصم غرامات التأخير من مستحقات المقاول دون الحاجة إلى إنذار",
"يلتزم المقاول بإنجاز الأعمال خلال ثلاثمئة وستين يوماً من تاريخ أمر المباشرة",
"تُحتسب الغرامة عن كل يوم تأخير بعد انقضاء مدة التنفيذ المعدلة",
"يحق لصاحب العمل سحب العمل إذا تجاوز التأخير تسعين يوماً",
"يقدّم المقاول برنامجاً زمنياً محدثاً كلما تأخر عن المسار الحرج",
"تُطبق تعويضات التأخير المقطوعة دون حاجة لإثبات الضرر الفعلي",
"يُمنح المقاول تمديداً للمدة إذا كان التأخير ناتجاً عن تعليمات صاحب العمل",
"تُفرض غرامة عن تأخر تسليم كل مرحلة من مراحل المشروع على حدة",
"لا يُعفى المقاول من الغرامة إلا بقرار خطي من صاحب العمل",
"يُعدّ تجاوز موعد الإنجاز المحدد إخلالاً جوهرياً بالعقد",
"يلتزم المقاول بتسريع العمل على نفقته لتعويض التأخير الحاصل",
"تُحسب مدة التأخير من تاريخ الإنجاز التعاقدي حتى تاريخ الاستلام الأولي",
"يدفع المقاول تعويضاً يومياً عن كل يوم يتأخر فيه تسليم المبنى",
"تبدأ مدة التنفيذ من اليوم التالي لاستلام الموقع خالياً من العوائق"],
"variation": [
"يحق للمهندس إصدار أوامر تغييرية بالزيادة أو النقصان في كميات الأعمال",
"لا يجوز للمقاول تنفيذ أي تعديل على المخططات دون أمر تغييري مكتوب",
"تُسعّر الأعمال المستجدة بأسعار مماثلة في جدول الكميات أو بأسعار يتفق عليها الطرفان",
"إذا تجاوزت الزيادة في الكميات خمسة وعشرين بالمئة يحق للطرفين إعادة التسعير",
"يقدّم المقاول عرض سعر للتعديل المطلوب خلال سبعة أيام من استلام التعليمات",
"يجوز لصاحب العمل حذف أي بند من الأعمال دون تعويض عن الربح الفائت",
"يُرفق بكل أمر تغييري بيان بأثره على مدة التنفيذ وقيمة العقد",
"تُعدّ التعليمات الشفوية لاغية ما لم تُؤكد خطياً خلال يومين",
"يلتزم المقاول بتنفيذ التعديلات التصميمية التي يطلبها الاستشاري",
"لا تتجاوز القيمة الإجمالية للأوامر التغييرية عشرين بالمئة من قيمة العقد دون موافقة الجهة المالكة",
"يُعاد احتساب المدة إذا أدى التعديل إلى زيادة الأعمال على المسار الحرج",
"تُعتمد الأعمال الإضافية بموجب محضر قياس موقّع من الطرفين",
"يجوز تغيير مواصفات المواد بشرط عدم انخفاض الجودة أو زيادة الكلفة",
"يحق للمقاول اقتراح بدائل هندسية توفر الكلفة ويقتسم الوفر مع صاحب العمل",
"كل تغيير في نطاق العمل يتطلب موافقة مسبقة من مدير المشروع",
"تُدرج الأعمال الملغاة في الحساب الختامي بالسالب"],
"force_majeure": [
"لا يُسأل أي من الطرفين عن الإخلال بالتزاماته إذا نتج عن قوة قاهرة",
"تشمل القوة القاهرة الحروب والزلازل والفيضانات والأوبئة",
"يُخطر الطرف المتضرر الطرف الآخر خلال أربعة عشر يوماً من وقوع الحدث القاهر",
"إذا استمرت القوة القاهرة أكثر من مئة وعشرين يوماً يحق لأي طرف إنهاء العقد",
"لا تُعدّ تقلبات أسعار المواد من قبيل القوة القاهرة",
"يُمدد موعد الإنجاز بقدر مدة توقف العمل بسبب الظروف الطارئة الخارجة عن الإرادة",
"الإضرابات العامة والقرارات الحكومية المفاجئة تُعامل معاملة الحدث القاهر",
"يبذل الطرف المتضرر جهده لتخفيف آثار الحدث الاستثنائي",
"لا يستحق المقاول تعويضاً مالياً عن فترة توقف القوة القاهرة بل تمديداً للمدة فقط",
"الأمطار الاستثنائية التي تتجاوز المعدل الموسمي تُعد ظرفاً قاهراً",
"يُرفق الإخطار بالأدلة التي تثبت وقوع الحدث وأثره على العمل",
"إذا انتهى العقد بسبب قوة قاهرة يُدفع للمقاول قيمة الأعمال المنفذة فقط",
"إغلاق الحدود ومنع الاستيراد يُعدان من الأحداث الخارجة عن سيطرة الطرفين",
"الجائحة الصحية التي تمنع وصول العمال إلى الموقع تُعد قوة قاهرة",
"لا يجوز التذرع بالقوة القاهرة إذا كان الحدث متوقعاً عند توقيع العقد",
"تُعلّق الالتزامات المتأثرة طوال مدة استمرار الظرف القاهر"],
"warranty": [
"يضمن المقاول الأعمال ضد العيوب لمدة سنة من تاريخ الاستلام الأولي",
"يلتزم المقاول بإصلاح أي عيب يظهر خلال فترة الصيانة على نفقته الخاصة",
"يكون المقاول والمصمم مسؤولين بالتضامن عن تهدم البناء خلال عشر سنوات",
"إذا لم يصلح المقاول العيب خلال المدة المحددة يحق لصاحب العمل إصلاحه على حساب المقاول",
"تُمدد فترة الصيانة للجزء الذي أُعيد إصلاحه",
"يقدّم المقاول ضمانات المصنّعين للمعدات الكهروميكانيكية لمدة لا تقل عن سنتين",
"تُجرى معاينة مشتركة قبل انتهاء فترة الصيانة بثلاثين يوماً",
"يضمن المقاول عزل الأسطح ضد تسرب المياه لمدة عشر سنوات",
"لا يصدر الاستلام النهائي قبل معالجة جميع الملاحظات المسجلة",
"تشمل المسؤولية العشرية العيوب التي تهدد متانة المبنى وسلامته",
"يلتزم المقاول بتسليم دليل التشغيل والصيانة قبل الاستلام الأولي",
"تُغطي كفالة حسن التنفيذ التزامات المقاول خلال فترة الضمان",
"يُعدّ ظهور تشققات إنشائية خلال فترة الضمان عيباً يستوجب الإصلاح الفوري",
"يلتزم المقاول بالاستجابة لبلاغات العيوب خلال ثمانٍ وأربعين ساعة",
"يتحمل المقاول كلفة الفحوص اللازمة لتحديد سبب العيب",
"تنتهي مسؤولية المقاول عن العيوب الظاهرة بصدور شهادة الاستلام النهائي"]}


In [ ]:
import numpy as np, matplotlib, warnings; warnings.filterwarnings("ignore")
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, roc_auc_score, roc_curve, confusion_matrix, accuracy_score
rng=np.random.default_rng(11)
# (1) cost per m2
n=900; year=rng.integers(2015,2026,n)
idx={y:100*(1.02**(min(y,2020)-2015))*(1.13 if y>=2021 else 1)*(1.12 if y>=2022 else 1)*(1.03**max(0,y-2022)) for y in range(2015,2026)}
pi=np.array([idx[y] for y in year]); q=rng.integers(1,4,n); loc=rng.choice([1.0,0.92,0.95,1.08],n)  # Amman,Zarqa,Irbid,Aqaba
area=rng.uniform(300,6000,n); floors=rng.integers(1,9,n)
cost=(180+70*(q-1))*loc*(pi/100)*(1+0.015*floors)*(area/1000)**-0.05*np.exp(rng.normal(0,0.07,n))
X=np.c_[year,q,loc,np.log(area),floors]; Xp=np.c_[np.log(pi),q,loc,np.log(area),floors]
tr,te=train_test_split(np.arange(n),test_size=0.25,random_state=0); old=year<=2021; new=~old
def ev(model,XX,a,b,logy=False):
    y=np.log(cost) if logy else cost; model.fit(XX[a],y[a]); p=model.predict(XX[b]); p=np.exp(p) if logy else p
    return mean_absolute_error(cost[b],p)/cost[b].mean()*100, p
r1,_=ev(GradientBoostingRegressor(random_state=0),X,tr,te); r2,pg=ev(GradientBoostingRegressor(random_state=0),X,old,new)
r3,pl=ev(LinearRegression(),Xp,old,new,logy=True)
print(f"GBM random split MAPE={r1:.1f}% ; GBM time split (train<=2021,test>=2022)={r2:.1f}% ; log-linear with price index time split={r3:.1f}%")
for yv in (2023,2025):
    s=year[new]==yv; print(yv,"mean actual",round(cost[new][s].mean()),"GBM",round(pg[s].mean()),"loglinear",round(pl[s].mean()))
fig,ax=plt.subplots(figsize=(6.5,3.4)); yrs=np.arange(2015,2026)
ax.plot(yrs,[cost[year==y].mean() for y in yrs],"o-",color="#2d3748",label="Actual mean")
ax.plot(yrs[yrs>=2022],[pg[year[new]==y].mean() for y in yrs[yrs>=2022]],"s--",color="#dd6b20",label="GBM (year as input)")
ax.plot(yrs[yrs>=2022],[pl[year[new]==y].mean() for y in yrs[yrs>=2022]],"^--",color="#2b6cb0",label="Log-linear with price index")
ax.axvspan(2021.5,2025.5,color="#edf2f7"); ax.text(2022,ax.get_ylim()[0]+5,"test period",fontsize=8)
ax.set_xlabel("Year"); ax.set_ylabel("Cost (JD/m2, illustrative)"); ax.legend(frameon=False,fontsize=8); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig/fig11_1_cost.png",dpi=200)
# (2) delay risk
m=1500; size=rng.uniform(np.log(0.2),np.log(50),m); design=rng.uniform(40,100,m); gov=rng.integers(0,2,m); lump=rng.integers(0,2,m); summer=rng.integers(0,2,m); var=rng.poisson(3,m)
z=-1.0+0.35*size-0.05*(design-70)+0.5*gov-0.3*lump+0.2*summer+0.25*var+0.02*(100-design)*gov
yd=(rng.random(m)<1/(1+np.exp(-z))).astype(int); D=np.c_[size,design,gov,lump,summer,var]
cv=StratifiedKFold(5,shuffle=True,random_state=0)
pL=cross_val_predict(LogisticRegression(max_iter=1000),D,yd,cv=cv,method="predict_proba")[:,1]
pG=cross_val_predict(GradientBoostingClassifier(random_state=0),D,yd,cv=cv,method="predict_proba")[:,1]
print(f"delayed share={yd.mean():.2f}; AUC logistic={roc_auc_score(yd,pL):.3f} GBM={roc_auc_score(yd,pG):.3f}")
lg=LogisticRegression(max_iter=1000).fit((D-D.mean(0))/D.std(0),yd); names=["log size","design %","government","lump sum","summer start","variations"]
print("standardised coefs:",dict(zip(names,np.round(lg.coef_[0],2))))
fig,ax=plt.subplots(figsize=(4.6,4.2))
for p,c,l in [(pL,"#2b6cb0","Logistic"),(pG,"#dd6b20","GBM")]:
    f,t,_=roc_curve(yd,p); ax.plot(f,t,color=c,lw=2,label=f"{l} AUC={roc_auc_score(yd,p):.2f}")
ax.plot([0,1],[0,1],color="#a0aec0",lw=1); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(frameon=False,fontsize=8); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig/fig11_2_roc.png",dpi=200)
# (3) clauses
from clauses import CLAUSES
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
txt=[s for k in CLAUSES for s in CLAUSES[k]]; lab=[k for k in CLAUSES for s in CLAUSES[k]]
clf=make_pipeline(TfidfVectorizer(analyzer="char_wb",ngram_range=(2,5)),LogisticRegression(max_iter=2000,C=10))
pc=cross_val_predict(clf,txt,lab,cv=StratifiedKFold(4,shuffle=True,random_state=0))
print(f"clauses={len(txt)} CV accuracy={accuracy_score(lab,pc):.3f}")
ks=list(CLAUSES); cm=confusion_matrix(lab,pc,labels=ks); print(cm)
wrong=[(t,l,p) for t,l,p in zip(txt,lab,pc) if l!=p]
for w in wrong[:6]: print("WRONG:",w)
clf.fit(txt,lab)
new=["تُصرف مستحقات المقاول خلال ستين يوماً من تقديم الفاتورة",
     "يُغرّم المقاول مبلغ مئتي دينار عن كل يوم تأخير في التسليم",
     "يجوز لصاحب العمل إضافة طابق إضافي بأمر خطي مع تعديل السعر",
     "يُعدّ الزلزال أو الحريق العام حدثاً خارجاً عن إرادة الطرفين",
     "يضمن المقاول سلامة الهيكل الإنشائي عشر سنوات من الاستلام"]
print("new clauses ->",list(clf.predict(new)))
fig,ax=plt.subplots(figsize=(5,4.2)); ax.imshow(cm,cmap="Blues")
lbl=["Payment","Delay","Variation","Force maj.","Warranty"]
ax.set_xticks(range(5),lbl,rotation=30,fontsize=8); ax.set_yticks(range(5),lbl,fontsize=8)
for i in range(5):
    for j in range(5): ax.text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>8 else "#2d3748")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.tight_layout(); fig.savefig("fig/fig11_3_clauses.png",dpi=200)